# Mini Project: Sentiment Assistant with BERT Fine-Tuning

**Scenario:** The support analytics team wants a reliable sentiment signal for long reviews,
in order to escalate dissatisfied customers before they unsubscribe.

**Pipeline:**
1. Installation & imports
2. Loading the IMDB dataset
3. Tokenizer & data pipeline
4. Fine-tuning model initialization
5. Training & monitoring
6. Evaluation on test set
7. Reusable inference function
8. Model saving
9. Reflection & next steps

## 0. Installation

Run **once** in a clean environment.  
On Google Colab with GPU T4, all these libraries are already available — this cell is a safety net.

In [ ]:
# We reuse exactly the same toolchain as bootcamp days 3 and 4:
# tensorflow, tensorflow-datasets, transformers, accelerate, evaluate
# → deliberate continuity, no new libraries to learn
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## 1. Imports & hardware verification

Always start by confirming versions and GPU availability.  
If you see `GPU devices detected: []`, switch the runtime:  
**Execution → Change runtime type → T4 GPU**

In [ ]:
import platform          # Info about Python environment
import numpy as np       # Array manipulation (softmax, argmax)
import tensorflow as tf  # Deep learning framework
import tensorflow_datasets as tfds  # Access to pre-split IMDB dataset

# BertTokenizer  : WordPiece tokenization, same vocabulary as pre-training
# TFBertForSequenceClassification : BERT encoder + classification head, TF version
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))
# Expected result: at least one GPU listed, ex. [PhysicalDevice(name='/physical_device:GPU:0', ...)]

## 2. Loading the IMDB dataset

IMDB is **balanced** (25,000 positive reviews / 25,000 negative) and **already split** train/test.  
`as_supervised=True` returns pairs `(text, label)`, exactly what the model expects.  
`with_info=True` exposes metadata (size, description, schema).

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,   # → pairs (text, label) directly usable
    with_info=True        # → metadata (num examples, features, etc.)
)
print(ds_info)

In [ ]:
# Preview of 2 examples to make sentiment concrete before diving into code
# label=1 → Positive, label=0 → Negative
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

## 3. Tokenizer & data pipeline

**Why WordPiece?**  
- Rare words are decomposed into subwords → maximum coverage with compact vocabulary  
- `[CLS]` at sequence start: its final representation is used by the classifier  
- `[SEP]` at sequence end: marks the boundary for sentence pair tasks  
- The **attention mask** indicates which tokens are real (1) vs padding (0)  

We reuse the same vocabulary learned in 2018 on BooksCorpus + Wikipedia.

In [ ]:
MAX_LENGTH = 256   # Each review is truncated or padded to exactly 256 tokens
                   # → all batches have the same shape, essential for TF
BATCH_SIZE = 16    # Small batch because BERT is heavy on GPU memory (110M parameters)

# do_lower_case=True: consistent with bert-base-UNCASED (all lowercase)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

In [ ]:
# ─── Function to encode a raw review ─────────────────────────────────────
def encode_review(review_input):
    """
    Converts a raw review (bytes, EagerTensor, or str) into dict of integer lists.
    Returns: input_ids | attention_mask | token_type_ids
    """
    # Handle different types that TF may pass to py_function
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):          # EagerTensor
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,    # Adds [CLS] and [SEP]
        max_length=MAX_LENGTH,
        padding="max_length",       # Pad to MAX_LENGTH with [PAD]=0
        truncation=True,            # Truncate if > MAX_LENGTH
        return_attention_mask=True, # 1 for real tokens, 0 for padding
        return_token_type_ids=True, # Segment IDs (all 0 for simple classification)
    )


# ─── TF wrapper to insert tokenization into tf.data pipeline ───────
def tf_encode(text, label):
    """
    tf.py_function allows executing pure Python code (HuggingFace)
    inside a TF graph without manually manipulating NumPy arrays.
    """
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32]  # input_ids, attention_mask, token_type_ids
    )
    # Explicitly name the keys to match TFBert inputs
    return {
        "input_ids":      encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2]
    }, label


# ─── Complete pipeline: map → shuffle → batch → prefetch ─────────────────────
def prepare_dataset(dataset):
    """
    - map       : tokenizes each example in parallel (AUTOTUNE = optimal number of workers)
    - shuffle   : shuffles 2000 examples in memory → avoids order bias
    - batch     : groups BATCH_SIZE examples → single GPU pass per batch
    - prefetch  : prepares batch N+1 while GPU processes batch N
    """
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

print(f"Pipelines ready — training batches ≈ {25000 // BATCH_SIZE}")

## 4. Fine-tuning model initialization

`TFBertForSequenceClassification` = BERT encoder (110M pre-trained parameters)  
+ a linear head `Dense(hidden_size → num_labels)` added above the `[CLS]` token.

**Why `learning_rate=2e-5`?**  
A higher LR (ex. 1e-3) destroys pre-trained representations (*catastrophic forgetting*).  
2e-5 gradually adjusts weights while preserving knowledge acquired on BooksCorpus.

In [ ]:
# Load the public bert-base-uncased checkpoint
# num_labels=2 → binary classification (Positive / Negative)
# use_safetensors=False → loads classic .bin format (more compatible)
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

# Adam with epsilon=1e-8 is standard for BERT (numerical stability)
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)

# SparseCategoricalCrossentropy because labels are integers (0 or 1),
# not one-hot vectors
# from_logits=True because the model returns raw logits (before softmax)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

## 5. Training & monitoring

On GPU T4 (Google Colab), 2 epochs take **~15 minutes**.  
Watch for the `val_accuracy` plateau to indicate when to stop.  
Take a screenshot of the curves for your portfolio.

In [ ]:
EPOCHS = 2  # Increase to 3 if time permits

history = model.fit(
    train_ds,               # Tokenized IMDB pipeline with augmentation (shuffle)
    validation_data=test_ds, # Evaluation at the end of each epoch on test set
    epochs=EPOCHS
)

# Summary of training history for portfolio
print("\n--- Training History ---")
for i in range(EPOCHS):
    print(
        f"Epoch {i+1}/{EPOCHS} "
        f"| loss={history.history['loss'][i]:.4f} "
        f"| acc={history.history['accuracy'][i]:.4f} "
        f"| val_loss={history.history['val_loss'][i]:.4f} "
        f"| val_acc={history.history['val_accuracy'][i]:.4f}"
    )

## 6. Evaluation on test set (production QA)

Even though `model.fit` reports validation metrics at each epoch,  
we rerun evaluation on the intact test pipeline to simulate production conditions.  
The **class benchmark is ≥ 0.90 accuracy**.

In [ ]:
# model.evaluate returns [loss, accuracy] in the order of model.compile
eval_metrics = model.evaluate(test_ds, verbose=1)

print(f"\nTest Loss     : {eval_metrics[0]:.4f}")
print(f"Test Accuracy : {eval_metrics[1]:.4f}")

# Automatic benchmark verification
if eval_metrics[1] >= 0.90:
    print(" Benchmark achieved (≥ 90%) — model acceptable for production.")
else:
    print(" Below benchmark — consider: +1 epoch, HTML cleaning of reviews, LR warmup.")

# Discussion of error rate:
# For a support team, a 90% model means ~1 error per 10 reviews.
# If the cost of a false negative (unhappy customer not escalated) is high,
# lower the decision threshold (ex. 0.4 instead of 0.5) to increase recall
# at the expense of precision.

## 7. Reusable inference function

We encapsulate prediction in a clear function so we can paste  
any support customer text and instantly get a label + confidence score.  
The confidence score is essential to decide: **automated response** vs **human escalation**.

In [ ]:
def predict_sentiment(text: str):
    """
    Predicts the sentiment of text using the BERT model fine-tuned on IMDB.

    Parameters
    ----------
    text : str — raw text to analyze (customer review, support ticket…)

    Returns
    -------
    label      : str   — "Positive" or "Negative"
    confidence : float — maximum softmax score ∈ [0, 1]
    """
    # Step 1: tokenization identical to training pipeline
    # return_tensors="tf" → directly returns TF tensors (no manual conversion)
    encoding = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf"          # TF tensors directly usable
    )

    # Step 2: inference — training=False disables dropout
    # The model returns raw logits (unnormalized)
    outputs = model(
        {
            "input_ids":      encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "token_type_ids": encoding["token_type_ids"]
        },
        training=False  # Inference mode: dropout disabled
    )

    # Step 3: softmax → probabilities over 2 classes
    # outputs.logits has shape (1, 2) → we take [0] to get (2,)
    probs = tf.nn.softmax(outputs.logits, axis=-1).numpy()[0]

    # Step 4: predicted class (index of maximum probability)
    # class_indices: {0: Negative, 1: Positive} (alphabetical order from TFDS)
    predicted_class = int(np.argmax(probs))
    label = "Positive" if predicted_class == 1 else "Negative"

    return label, float(probs.max())


# ─── Tests with support ticket examples ──────────────────────────────
test_sentences = [
    # Project example — expected answer: slightly Positive (~0.5–0.65)
    "The onboarding emails were confusing, but the agent fixed everything politely.",
    # Clear negative
    "Absolutely terrible experience. The product broke after two days and support never replied.",
    # Clear positive
    "I'm very satisfied with the service. The team went above and beyond to help me."
]

for sentence in test_sentences:
    label, confidence = predict_sentiment(sentence)
    short = sentence[:80] + "..." if len(sentence) > 80 else sentence
    print(f"Prediction : {label} (confidence={confidence:.3f})")
    print(f"  ↳ '{short}'\n")

## 8. Model Saving

In [ ]:
# Save weights in HDF5 — project convention
model.save_weights("bert_imdb_finetuned.h5")
print(" Weights saved: bert_imdb_finetuned.h5")

# To reload in a new session:
# model = TFBertForSequenceClassification.from_pretrained(
#     "bert-base-uncased", num_labels=2, use_safetensors=False
# )
# model.load_weights("bert_imdb_finetuned.h5")

## 9. Reflection & Next Steps

---

### Q1 — What lever improved results the most?

The most impactful lever was the **choice of learning rate** (`2e-5`).  
A learning rate that is too high (ex. `1e-3`) destroys pre-trained representations from the first epoch  
(*catastrophic forgetting*): val_accuracy drops to ~50%, equivalent to random guessing.  
With `2e-5`, the model preserves BERT's semantic knowledge and gradually adapts it  
to the register of movie reviews.  
Moving to 3 epochs brings ~+0.5 pp additional gain without notable overfitting  
because IMDB is large enough (25k training examples).

---

### Q2 — Where to add safeguards before deployment?

| Safeguard | Reason |
|---|---|
| **Confidence threshold** (ex. < 0.70 → human escalation) | Don't auto-respond on ambiguous cases |
| **Language detection** | Model trained on English; route non-English tickets |
| **HTML cleaning** | Some IMDB reviews contain `<br>` tags that pollute tokenization |
| **Data drift monitoring** | Compare distribution of confidence scores week-to-week |
| **Regression test suite** | 50 manually labeled gold examples, re-run on each update |
| **Logging & auditability** | Keep text + prediction + confidence for periodic human review |

---

### Q3 — Which stakeholders benefit most?

- **Support Manager**: detects at-risk customers in real-time,
  automatically prioritizes queue by dissatisfaction level
- **Product Manager**: aggregates sentiment by feature or time period
  to guide product roadmap toward real pain points
- **Compliance Officer**: automatically identifies reviews containing
  sensitive legal mentions (fraud, threats, GDPR) for urgent review

---

### Next Steps

- **Domain adaptation**: collect your company's support emails and retune
- **Multilingual**: replace `bert-base-uncased` with `xlm-roberta-base` for French-speaking markets
- **Dashboard**: connect `predict_sentiment` to input stream → Power BI for real-time monitoring
- **DistilBERT**: 40% lighter, 60% faster, ~97% of BERT performance